In [1]:
import geopandas as gpd
import numpy as np
import pandas as pd
import swifter
import time
import warnings

from src import meteorological_api
import src.utils as utils

In [2]:
warnings.filterwarnings('ignore')

In [3]:
response_var = pd.read_parquet('../data/interim/storm_outages_2014_2023.parquet')
counties = gpd.read_parquet('../data/external/county.parquet')

In [4]:
counties['fips_code_id'] = counties['STATEFP'] + counties['COUNTYFP'] 
outages_storms = response_var[response_var.storm_caused_outage==1]

In [5]:
random_seed = 42

In [6]:
def get_url_params_from_row_point(storm_outage_row, desired_point=None):
    lon = storm_outage_row[desired_point].x
    lat = storm_outage_row[desired_point].y
    start = storm_outage_row.one_day_before_storm
    end = storm_outage_row.one_day_after_storm
    identifier = storm_outage_row['episode_fips_id']
    save_path = f'../data/raw/meteorological/{identifier}.json'
    filexists = utils.check_if_filepath_exists(save_path)
    if not filexists:
        url = meteorological_api.get_api_url(lat=lat, lon=lon, datetime_start=start, datetime_end=end)
        return url
    return np.nan

def save_meteorological_info_from_row_point(storm_outage_row_iterated):
    idx, storm_outage_row = storm_outage_row_iterated
    identifier = storm_outage_row['episode_fips_id']
    meteorological_info = storm_outage_row['meteorological_info__cntroid']
    validate = meteorological_info.get('properties')
    save_path = f'../data/raw/meteorological/{identifier}.json'
    filexists = utils.check_if_filepath_exists(save_path)
    if (not filexists) and (validate is not None):
        meteorological_api.save_meteorological_information(information=meteorological_info, path=save_path)
        #print(f'Saving info at: {save_path}')
        return 1
    return np.nan

In [7]:
element_nb = 2200

In [318]:
condition = 100

In [320]:
while condition:
    # Transform data
    print('Starting')
    element_nb += 200
    sample_outages_storms = outages_storms.sample(element_nb, random_state=random_seed)
    meaning_dict = {'begin_datetime': 'storm_start', 'end_datetime': 'storm_end', 'run_start_time_min': 'outage_start', 'run_start_time_max': 'outage_end'}
    sample_outages_storms = sample_outages_storms.rename(
        columns=meaning_dict
    )
    sample_outages_storms['one_day_before_storm'] = (sample_outages_storms['storm_start'] - pd.Timedelta(1, unit='D')).dt.date
    sample_outages_storms['one_day_after_storm'] = (sample_outages_storms['storm_end'] + pd.Timedelta(1, unit='D')).dt.date
    sample_outages_storms = sample_outages_storms.merge(counties[['fips_code_id', 'geometry']], on='fips_code_id')
    sample_outages_storms = gpd.GeoDataFrame(sample_outages_storms)
    sample_outages_storms['cntroid'] = sample_outages_storms.geometry.centroid
    sample_outages_storms['url__cntroid'] = sample_outages_storms.apply(get_url_params_from_row_point, desired_point='cntroid', axis=1)
    sample_outages_storms = sample_outages_storms[sample_outages_storms['url__cntroid'].notna()]
        # Request meteorological data
    t0= time.perf_counter()
    sample_outages_storms['meteorological_info__cntroid'] = sample_outages_storms.url__cntroid.swifter.allow_dask_on_strings(
        enable=True
    ).apply(
        meteorological_api.get_meteorological_info
    )
    t1 = time.perf_counter()
    print(f'time it took {t1-t0}')
    # Save data
    for row in sample_outages_storms.iterrows():
        save_meteorological_info_from_row_point(row)
    print('Done with saving.')
    # Wait.
    time.sleep(30)
    condition = condition - 1
    # Repeat.

Starting


Dask Apply:   0%|          | 0/41 [00:00<?, ?it/s]

/mnt/c/Users/52333/Documents/projects/dynamic-rythms/venv/lib/python3.10/site-packages/dask/utils.py:1226: FutureWarning: the convert_dtype parameter is deprecated and will be removed in a future version.  Do ``ser.astype(object).apply()`` instead if you want ``convert_dtype=False``.
  return getattr(__obj, self.method)(*args, **kwargs)
/mnt/c/Users/52333/Documents/projects/dynamic-rythms/venv/lib/python3.10/site-packages/dask/utils.py:1226: FutureWarning: the convert_dtype parameter is deprecated and will be removed in a future version.  Do ``ser.astype(object).apply()`` instead if you want ``convert_dtype=False``.
  return getattr(__obj, self.method)(*args, **kwargs)
/mnt/c/Users/52333/Documents/projects/dynamic-rythms/venv/lib/python3.10/site-packages/dask/utils.py:1226: FutureWarning: the convert_dtype parameter is deprecated and will be removed in a future version.  Do ``ser.astype(object).apply()`` instead if you want ``convert_dtype=False``.
  return getattr(__obj, self.method)(

time it took 1603.9848179580003
Saving info at: ../data/raw/meteorological/103872_25017.json
Saving info at: ../data/raw/meteorological/141313_15003.json
Saving info at: ../data/raw/meteorological/168421_48085.json
Saving info at: ../data/raw/meteorological/121858_28035.json
Saving info at: ../data/raw/meteorological/149876_19153.json
Saving info at: ../data/raw/meteorological/150658_06013.json
Saving info at: ../data/raw/meteorological/155465_48085.json
Saving info at: ../data/raw/meteorological/155768_53061.json
Saving info at: ../data/raw/meteorological/103927_40099.json
Saving info at: ../data/raw/meteorological/99352_06065.json
Saving info at: ../data/raw/meteorological/125586_46099.json
Saving info at: ../data/raw/meteorological/182296_48067.json
Saving info at: ../data/raw/meteorological/168039_28011.json
Saving info at: ../data/raw/meteorological/167527_22051.json
Saving info at: ../data/raw/meteorological/155465_48363.json
Saving info at: ../data/raw/meteorological/110691_3903

Dask Apply:   0%|          | 0/41 [00:00<?, ?it/s]

...

time it took 838.3517716439965
Saving info at: ../data/raw/meteorological/179821_51087.json
Saving info at: ../data/raw/meteorological/94951_55085.json
Saving info at: ../data/raw/meteorological/122640_34013.json
Saving info at: ../data/raw/meteorological/167449_27025.json
Saving info at: ../data/raw/meteorological/162695_04019.json
Saving info at: ../data/raw/meteorological/155121_25021.json
Saving info at: ../data/raw/meteorological/129475_37081.json
Saving info at: ../data/raw/meteorological/155803_40041.json
Saving info at: ../data/raw/meteorological/110820_06073.json
Saving info at: ../data/raw/meteorological/147983_48265.json
Saving info at: ../data/raw/meteorological/103788_48339.json
Saving info at: ../data/raw/meteorological/111642_06073.json
Saving info at: ../data/raw/meteorological/137435_55097.json
Saving info at: ../data/raw/meteorological/111316_51800.json
Saving info at: ../data/raw/meteorological/144001_48453.json
Saving info at: ../data/raw/meteorological/176711_55075

Dask Apply:   0%|          | 0/41 [00:00<?, ?it/s]

/mnt/c/Users/52333/Documents/projects/dynamic-rythms/venv/lib/python3.10/site-packages/dask/utils.py:1226: FutureWarning: the convert_dtype parameter is deprecated and will be removed in a future version.  Do ``ser.astype(object).apply()`` instead if you want ``convert_dtype=False``.
  return getattr(__obj, self.method)(*args, **kwargs)
/mnt/c/Users/52333/Documents/projects/dynamic-rythms/venv/lib/python3.10/site-packages/dask/utils.py:1226: FutureWarning: the convert_dtype parameter is deprecated and will be removed in a future version.  Do ``ser.astype(object).apply()`` instead if you want ``convert_dtype=False``.
  return getattr(__obj, self.method)(*args, **kwargs)
/mnt/c/Users/52333/Documents/projects/dynamic-rythms/venv/lib/python3.10/site-packages/dask/utils.py:1226: FutureWarning: the convert_dtype parameter is deprecated and will be removed in a future version.  Do ``ser.astype(object).apply()`` instead if you want ``convert_dtype=False``.
  return getattr(__obj, self.method)(

time it took 821.6842419570021
Saving info at: ../data/raw/meteorological/112323_25015.json
Saving info at: ../data/raw/meteorological/174634_06071.json
Saving info at: ../data/raw/meteorological/181325_40027.json
Saving info at: ../data/raw/meteorological/172821_34029.json
Saving info at: ../data/raw/meteorological/187305_48085.json
Saving info at: ../data/raw/meteorological/162266_06065.json
Saving info at: ../data/raw/meteorological/186630_48455.json
Saving info at: ../data/raw/meteorological/129369_34039.json
Saving info at: ../data/raw/meteorological/155546_06029.json
Saving info at: ../data/raw/meteorological/146304_28075.json
Saving info at: ../data/raw/meteorological/137435_55073.json
Saving info at: ../data/raw/meteorological/184005_45083.json
Saving info at: ../data/raw/meteorological/158786_12011.json
Saving info at: ../data/raw/meteorological/147846_28137.json
Saving info at: ../data/raw/meteorological/148467_48473.json
Saving info at: ../data/raw/meteorological/102566_1000

KeyboardInterrupt: 

In [313]:
# This is not needed...

In [309]:
sample_outages_storms = outages_storms.sample(element_nb, random_state=random_seed)
meaning_dict = {'begin_datetime': 'storm_start', 'end_datetime': 'storm_end', 'run_start_time_min': 'outage_start', 'run_start_time_max': 'outage_end'}
sample_outages_storms = sample_outages_storms.rename(
    columns=meaning_dict
)
sample_outages_storms['one_day_before_storm'] = (sample_outages_storms['storm_start'] - pd.Timedelta(1, unit='D')).dt.date
sample_outages_storms['one_day_after_storm'] = (sample_outages_storms['storm_end'] + pd.Timedelta(1, unit='D')).dt.date
sample_outages_storms = sample_outages_storms.merge(counties[['fips_code_id', 'geometry']], on='fips_code_id')
sample_outages_storms = gpd.GeoDataFrame(sample_outages_storms)

In [310]:
sample_outages_storms['rep_point'] = sample_outages_storms.geometry.representative_point()
#sample_outages_storms['up'] = sample_outages_storms.extract_unique_points()
sample_outages_storms['cntroid'] = sample_outages_storms.geometry.centroid
sample_outages_storms['boundary_sample_p'] = sample_outages_storms.geometry.boundary.sample_points(10, random_seed=random_seed)
sample_outages_storms['sample_p1'] = sample_outages_storms.geometry.sample_points(size=1, random_seed=random_seed)
sample_outages_storms['sample_p2'] = sample_outages_storms.geometry.sample_points(size=1, random_seed=random_seed+1)
sample_outages_storms['sample_p3'] = sample_outages_storms.geometry.sample_points(size=1, random_seed=random_seed+2)

In [311]:
# #!pip install pointpats
# s = sample_outages_storms.sample(1)
# ax = s.plot()
# s.rep_point.plot(ax=ax, color='r')
# #s.up.plot(ax=ax, color='g')
# s.cntroid.plot(ax=ax, color='y')
# s.boundary_sample_p.plot(ax=ax, color='orange')
# s.sample_p1.plot(ax=ax, color='gray')
# s.sample_p2.plot(ax=ax, color='gray')
# s.sample_p3.plot(ax=ax, color='gray')

In [291]:

    
#sample_outages_storms['url__rep_point'] = sample_outages_storms.apply(get_url_params_from_row_point, desired_point='rep_point', axis=1)
sample_outages_storms['url__cntroid'] = sample_outages_storms.apply(get_url_params_from_row_point, desired_point='cntroid', axis=1)
#sample_outages_storms['url__sp1'] = sample_outages_storms.apply(get_url_params_from_row_point, desired_point='sample_p1', axis=1)
#sample_outages_storms['url__sp2'] = sample_outages_storms.apply(get_url_params_from_row_point, desired_point='sample_p2', axis=1)
#sample_outages_storms['url__sp3'] = sample_outages_storms.apply(get_url_params_from_row_point, desired_point='sample_p3', axis=1)

In [292]:
sample_outages_storms = sample_outages_storms[sample_outages_storms['url__cntroid'].notna()]

In [293]:
#sample_outages_storms.url__rep_point.iloc[0]

In [294]:
sample_outages_storms

,EPISODE_ID,fips_code_id,episode_description,storm_start,storm_end,storm_duration,episode_fips_id,storm_caused_outage,outage_index_id,outage_start_minus_storm_start,...,one_day_before_storm,one_day_after_storm,geometry,rep_point,cntroid,boundary_sample_p,sample_p1,sample_p2,sample_p3,url__cntroid
1134,177512,41007,"Combination of snow, sleet and freezing rain.",2022-12-22 16:00:00,2022-12-24 13:30:00,45.500000,177512_41007,1.0,41007__0028,1.104167,...,2022-12-21,2022-12-25,"POLYGON ((-124.01136 46.23622, -124.002 46.237...",POINT (-123.64603 46.02537),POINT (-123.65572 45.99514),"MULTIPOINT (-123.99276 45.94733, -123.98143 46...",POINT (-123.61519 46.16772),POINT (-123.80363 45.82972),POINT (-123.45997 46.25325),https://power.larc.nasa.gov/api/temporal/hourl...
1297,143238,48491,Nearly all of South Central Texas had less tha...,2019-10-01 00:00:00,2019-10-31 23:59:00,743.983333,143238_48491,1.0,48491__0043,24.010417,...,2019-09-30,2019-11-01,"POLYGON ((-98.04989 30.62416, -97.998 30.72021...",POINT (-97.63453 30.66086),POINT (-97.60028 30.64775),"MULTIPOINT (-97.92403 30.60898, -97.86906 30.5...",POINT (-97.27337 30.62153),POINT (-97.32981 30.50738),POINT (-97.72788 30.59015),https://power.larc.nasa.gov/api/temporal/hourl...
1298,133333,39045,Several waves of low pressure moved along a st...,2019-02-06 11:48:00,2019-02-08 01:40:00,37.866667,133333_39045,1.0,39045__0026,1.497917,...,2019-02-05,2019-02-09,"POLYGON ((-82.83646 39.62786, -82.83549 39.637...",POINT (-82.61561 39.75553),POINT (-82.63066 39.75161),"MULTIPOINT (-82.83847 39.60729, -82.83312 39.6...",POINT (-82.54017 39.63577),POINT (-82.64963 39.88632),POINT (-82.61747 39.70291),https://power.larc.nasa.gov/api/temporal/hourl...
1323,122191,01123,A significant snowstorm impacted central Alaba...,2017-12-08 06:00:00,2017-12-09 05:00:00,23.000000,122191_01123,1.0,01123__0000,1.281250,...,2017-12-07,2017-12-10,"POLYGON ((-86.00917 33.09026, -85.97456 33.090...",POINT (-85.7985 32.79904),POINT (-85.79737 32.86253),"MULTIPOINT (-86.00765 32.84728, -86.00273 32.8...",POINT (-85.74843 32.91815),POINT (-85.62518 32.82237),POINT (-85.90016 32.8075),https://power.larc.nasa.gov/api/temporal/hourl...
1324,140999,55131,Persistent southwest winds and the passage of ...,2019-07-19 09:00:00,2019-07-20 14:20:00,29.333333,140999_55131,1.0,55131__0012,1.385417,...,2019-07-18,2019-07-21,"POLYGON ((-88.41871 43.32544, -88.41856 43.365...",POINT (-88.22326 43.36869),POINT (-88.2306 43.36852),"MULTIPOINT (-88.41858 43.23706, -88.40971 43.1...",POINT (-88.18444 43.53001),POINT (-88.22479 43.37107),POINT (-88.34135 43.41851),https://power.larc.nasa.gov/api/temporal/hourl...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1685,162889,04021,A shortwave trough rotating through the region...,2021-08-16 22:20:00,2021-08-17 09:00:00,10.666667,162889_04021,1.0,04021__0023,0.309028,...,2021-08-15,2021-08-18,"POLYGON ((-112.20359 32.63781, -112.20352 32.8...",POINT (-111.32618 32.91065),POINT (-111.34481 32.90444),"MULTIPOINT (-112.19055 33.26471, -112.03076 33...",POINT (-111.88107 32.64663),POINT (-111.93212 32.94542),POINT (-111.88854 32.9753),https://power.larc.nasa.gov/api/temporal/hourl...
1686,132631,39017,An upper level low pressure center tracked nor...,2018-11-14 23:00:00,2018-11-15 12:00:00,13.000000,132631_39017,1.0,39017__0043,0.406250,...,2018-11-13,2018-11-16,"POLYGON ((-84.81935 39.30945, -84.81745 39.391...",POINT (-84.57696 39.44808),POINT (-84.57555 39.43865),"MULTIPOINT (-84.81544 39.51715, -84.81214 39.5...",POINT (-84.45194 39.41688),POINT (-84.44585 39.40869),POINT (-84.36739 39.39598),https://power.larc.nasa.gov/api/temporal/hourl...
1687,99672,29113,A frontal boundary was slowly sinking south ac...,2015-06-25 17:25:00,2015-06-26 14:55:00,21.500000,99672_29113,1.0,29113__0000,0.305556,...,2015-06-24,2015-06-27,"POLYGON ((-91.26311 39.037, -91.26023 39.13984...",POINT (-90.98767 39.05039),POINT (-90.95986 39.05802),"MULTIPOINT (-91.18894 38

In [295]:
sample_outages_storms.url__cntroid.iloc[0]

'https://power.larc.nasa.gov/api/temporal/hourly/point?parameters=T2M,ALLSKY_SFC_SW_DWN,PS,WS50M,ALLSKY_SFC_LW_DWN,WD50M,PRECTOTCORR,GWETPROF,CLRSKY_SFC_SW_DWN,QV10M,RHOA,T10M,TO3,TQV,Z0M,TOA_SW_DWN,RH2M,WS2M,CLRSKY_SFC_LW_DWN,DISPH&community=RE&longitude=-123.6557206172946&latitude=45.995139447263576&start=20221221&end=20221225&format=JSON'

In [296]:
#sample_outages_storms.url__sp1.iloc[0]

In [297]:
#%%time
#sample_outages_storms['meteorological_info__rep_point'] = sample_outages_storms.url__rep_point.apply(meteorological_api.get_meteorological_info)

In [298]:
#%%time
#sample_outages_storms['meteorological_info__url__cntroid'] = sample_outages_storms.url__cntroid.apply(meteorological_api.get_meteorological_info)

In [299]:
%%time
sample_outages_storms['meteorological_info__cntroid'] = sample_outages_storms.url__cntroid.swifter.allow_dask_on_strings(enable=True).apply(meteorological_api.get_meteorological_info)

Dask Apply:   0%|          | 0/41 [00:00<?, ?it/s]

/mnt/c/Users/52333/Documents/projects/dynamic-rythms/venv/lib/python3.10/site-packages/dask/utils.py:1226: FutureWarning: the convert_dtype parameter is deprecated and will be removed in a future version.  Do ``ser.astype(object).apply()`` instead if you want ``convert_dtype=False``.
  return getattr(__obj, self.method)(*args, **kwargs)
/mnt/c/Users/52333/Documents/projects/dynamic-rythms/venv/lib/python3.10/site-packages/dask/utils.py:1226: FutureWarning: the convert_dtype parameter is deprecated and will be removed in a future version.  Do ``ser.astype(object).apply()`` instead if you want ``convert_dtype=False``.
  return getattr(__obj, self.method)(*args, **kwargs)
/mnt/c/Users/52333/Documents/projects/dynamic-rythms/venv/lib/python3.10/site-packages/dask/utils.py:1226: FutureWarning: the convert_dtype parameter is deprecated and will be removed in a future version.  Do ``ser.astype(object).apply()`` instead if you want ``convert_dtype=False``.
  return getattr(__obj, self.method)(

CPU times: user 770 ms, sys: 182 ms, total: 953 ms
Wall time: 14min 7s


In [300]:
#sample_outages_storms['meteorological_info__cntroid']

In [301]:
#rep_point = pd.DataFrame(sample_outages_storms.meteorological_info__rep_point.iloc[9]['properties']['parameter'])
centroid = pd.DataFrame(sample_outages_storms.meteorological_info__cntroid.iloc[9]['properties']['parameter'])
#rep_point-centroid


In [302]:
#sample_outages_storms['meteorological_info__cntroid'].iloc[0].get('properties')

In [303]:
def save_meteorological_info_from_row_point(storm_outage_row_iterated):
    idx, storm_outage_row = storm_outage_row_iterated
    identifier = storm_outage_row['episode_fips_id']
    meteorological_info = storm_outage_row['meteorological_info__cntroid']
    validate = meteorological_info.get('properties')
    save_path = f'../data/raw/meteorological/{identifier}.json'
    filexists = utils.check_if_filepath_exists(save_path)
    if (not filexists) and (validate is not None):
        meteorological_api.save_meteorological_information(information=meteorological_info, path=save_path)
        print(f'Saving info at: {save_path}')
        return 1
    return np.nan

In [304]:
for row in sample_outages_storms.iterrows():
    save_meteorological_info_from_row_point(row)

Saving info at: ../data/raw/meteorological/177512_41007.json
Saving info at: ../data/raw/meteorological/143238_48491.json
Saving info at: ../data/raw/meteorological/133333_39045.json
Saving info at: ../data/raw/meteorological/122191_01123.json
Saving info at: ../data/raw/meteorological/157072_48273.json
Saving info at: ../data/raw/meteorological/164104_06067.json
Saving info at: ../data/raw/meteorological/155446_48005.json
Saving info at: ../data/raw/meteorological/112444_12109.json
Saving info at: ../data/raw/meteorological/97063_24033.json
Saving info at: ../data/raw/meteorological/133253_17197.json
Saving info at: ../data/raw/meteorological/93589_21111.json
Saving info at: ../data/raw/meteorological/148533_39165.json
Saving info at: ../data/raw/meteorological/126901_12021.json
Saving info at: ../data/raw/meteorological/184949_49049.json
Saving info at: ../data/raw/meteorological/164151_06087.json
Saving info at: ../data/raw/meteorological/99116_48005.json
Saving info at: ../data/raw

In [305]:
sample_outages_storms

,EPISODE_ID,fips_code_id,episode_description,storm_start,storm_end,storm_duration,episode_fips_id,storm_caused_outage,outage_index_id,outage_start_minus_storm_start,...,one_day_after_storm,geometry,rep_point,cntroid,boundary_sample_p,sample_p1,sample_p2,sample_p3,url__cntroid,meteorological_info__cntroid
1134,177512,41007,"Combination of snow, sleet and freezing rain.",2022-12-22 16:00:00,2022-12-24 13:30:00,45.500000,177512_41007,1.0,41007__0028,1.104167,...,2022-12-25,"POLYGON ((-124.01136 46.23622, -124.002 46.237...",POINT (-123.64603 46.02537),POINT (-123.65572 45.99514),"MULTIPOINT (-123.99276 45.94733, -123.98143 46...",POINT (-123.61519 46.16772),POINT (-123.80363 45.82972),POINT (-123.45997 46.25325),https://power.larc.nasa.gov/api/temporal/hourl...,"{'type': 'Feature', 'geometry': {'type': 'Poin..."
1297,143238,48491,Nearly all of South Central Texas had less tha...,2019-10-01 00:00:00,2019-10-31 23:59:00,743.983333,143238_48491,1.0,48491__0043,24.010417,...,2019-11-01,"POLYGON ((-98.04989 30.62416, -97.998 30.72021...",POINT (-97.63453 30.66086),POINT (-97.60028 30.64775),"MULTIPOINT (-97.92403 30.60898, -97.86906 30.5...",POINT (-97.27337 30.62153),POINT (-97.32981 30.50738),POINT (-97.72788 30.59015),https://power.larc.nasa.gov/api/temporal/hourl...,"{'type': 'Feature', 'geometry': {'type': 'Poin..."
1298,133333,39045,Several waves of low pressure moved along a st...,2019-02-06 11:48:00,2019-02-08 01:40:00,37.866667,133333_39045,1.0,39045__0026,1.497917,...,2019-02-09,"POLYGON ((-82.83646 39.62786, -82.83549 39.637...",POINT (-82.61561 39.75553),POINT (-82.63066 39.75161),"MULTIPOINT (-82.83847 39.60729, -82.83312 39.6...",POINT (-82.54017 39.63577),POINT (-82.64963 39.88632),POINT (-82.61747 39.70291),https://power.larc.nasa.gov/api/temporal/hourl...,"{'type': 'Feature', 'geometry': {'type': 'Poin..."
1323,122191,01123,A significant snowstorm impacted central Alaba...,2017-12-08 06:00:00,2017-12-09 05:00:00,23.000000,122191_01123,1.0,01123__0000,1.281250,...,2017-12-10,"POLYGON ((-86.00917 33.09026, -85.97456 33.090...",POINT (-85.7985 32.79904),POINT (-85.79737 32.86253),"MULTIPOINT (-86.00765 32.84728, -86.00273 32.8...",POINT (-85.74843 32.91815),POINT (-85.62518 32.82237),POINT (-85.90016 32.8075),https://power.larc.nasa.gov/api/temporal/hourl...,"{'type': 'Feature', 'geometry': {'type': 'Poin..."
1324,140999,55131,Persistent southwest winds and the passage of ...,2019-07-19 09:00:00,2019-07-20 14:20:00,29.333333,140999_55131,1.0,55131__0012,1.385417,...,2019-07-21,"POLYGON ((-88.41871 43.32544, -88.41856 43.365...",POINT (-88.22326 43.36869),POINT (-88.2306 43.36852),"MULTIPOINT (-88.41858 43.23706, -88.40971 43.1...",POINT (-88.18444 43.53001),POINT (-88.22479 43.37107),POINT (-88.34135 43.41851),https://power.larc.nasa.gov/api/temporal/hourl...,{'message': 'Internal server error'}
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1685,162889,04021,A shortwave trough rotating through the region...,2021-08-16 22:20:00,2021-08-17 09:00:00,10.666667,162889_04021,1.0,04021__0023,0.309028,...,2021-08-18,"POLYGON ((-112.20359 32.63781, -112.20352 32.8...",POINT (-111.32618 32.91065),POINT (-111.34481 32.90444),"MULTIPOINT (-112.19055 33.26471, -112.03076 33...",POINT (-111.88107 32.64663),POINT (-111.93212 32.94542),POINT (-111.88854 32.9753),https://power.larc.nasa.gov/api/temporal/hourl...,"{'type': 'Feature', 'geometry': {'type': 'Poin..."
1686,132631,39017,An upper level low pressure center tracked nor...,2018-11-14 23:00:00,2018-11-15 12:00:00,13.000000,132631_39017,1.0,39017__0043,0.406250,...,2018-11-16,"POLYGON ((-84.81935 39.30945, -84.81745 39.391...",POINT (-84.57696 39.44808),POINT (-84.57555 39.43865),"MULTIPOINT (-84.81544 39.51715, -84.81214 39.5...",POINT (-84.45194 39.41688),POINT (-84.44585 39.40869),POINT (-84.36739 39.39598),https://power.larc.nasa.gov/api/temporal/hourl...,"{'type': 'Feature', 'geometry': {'type': 'Poin..."
1687,99672,29113,A frontal boun